# Download and Data Pre-processing

## **Download the UrbanSound8k Dataset**

Links used to download dataset: 
- https://urbansounddataset.weebly.com/urbansound8k.html#10foldCV
- https://github.com/soundata/soundata#quick-example

In [6]:
pip install soundata

Note: you may need to restart the kernel to use updated packages.


In [7]:
import soundata

dataset = soundata.initialize('urbansound8k')
dataset.download()  # download the dataset
dataset.validate()  # validate that all the expected files are there

example_clip = dataset.choice_clip()  # choose a random example clip
print(example_clip)  # see the available data

INFO: Downloading ['all', 'index']. Index is being stored in C:\Users\anada\AppData\Local\Programs\Python\Python310\Lib\site-packages\soundata\datasets\indexes, and the rest of files in /tmp\sound_datasets\urbansound8k
INFO: [all] downloading UrbanSound8K.tar.gz
INFO: /tmp\sound_datasets\urbansound8k\UrbanSound8K.tar.gz already exists and will not be downloaded. Rerun with force_overwrite=True to delete this file and force the download.
INFO: /tmp\sound_datasets\urbansound8k\audio already exists. Run with force_overwrite=True to download from scratch
INFO: /tmp\sound_datasets\urbansound8k\FREESOUNDCREDITS.txt already exists. Run with force_overwrite=True to download from scratch
INFO: /tmp\sound_datasets\urbansound8k\metadata already exists. Run with force_overwrite=True to download from scratch
INFO: /tmp\sound_datasets\urbansound8k\UrbanSound8K_README.txt already exists. Run with force_overwrite=True to download from scratch
INFO: [index] downloading urbansound8k_index_1.0.json
INFO:

Clip(
  audio_path="/tmp\sound_datasets\urbansound8k\audio/fold2/14780-9-0-2.wav",
  clip_id="14780-9-0-2",
  audio: The clip's audio
            * np.ndarray - audio signal
            * float - sample rate,
  class_id: The clip's class id.
            * int - integer representation of the class label (0-9). See Dataset Info in the documentation for mapping,
  class_label: The clip's class label.
            * str - string class name: air_conditioner, car_horn, children_playing, dog_bark, drilling, engine_idling, gun_shot, jackhammer, siren, street_music,
  fold: The clip's fold.
            * int - fold number (1-10) to which this clip is allocated. Use these folds for cross validation,
  freesound_end_time: The clip's end time in Freesound.
            * float - end time in seconds of the clip in the original freesound recording,
  freesound_id: The clip's Freesound ID.
            * str - ID of the freesound.org recording from which this clip was taken,
  freesound_start_time: The 

## **Data pre-processing and preparation**

### Import Libraries

In [8]:
import os
import librosa
import numpy as np
import pandas as pd

### Parameters

To choose these parameters we followed: https://www.justinsalamon.com/uploads/4/3/9/4/4394963/salamon_urbansound_acmmm14.pdf

In [18]:
SAMPLE_RATE = 22050        # Sampling rate for all clips
DURATION = 4               # All clips will be 4 seconds
SAMPLES_PER_CLIP = int(SAMPLE_RATE * DURATION)

# Features for RNN
N_MFCC = 25

# Features for CNN 
# N_MELS = 

N_FFT = 1024
HOP_LENGTH = 512

### Load the Dataset

In [19]:
dataset_path = dataset.data_home
metadata_path = os.path.join(dataset_path, "metadata", "UrbanSound8K.csv")
metadata = pd.read_csv(metadata_path)

def get_file_path(row):
    fold = f"fold{row['fold']}"
    file_name = row["slice_file_name"]
    return os.path.join(dataset_path, "audio", fold, file_name)

metadata["file_path"] = metadata.apply(get_file_path, axis=1)
metadata.head()

,slice_file_name,fsID,start,end,salience,fold,classID,class,file_path
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark,/tmp\sound_datasets\urbansound8k\audio\fold5\1...
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing,/tmp\sound_datasets\urbansound8k\audio\fold5\1...
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing,/tmp\sound_datasets\urbansound8k\audio\fold5\1...
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing,/tmp\sound_datasets\urbansound8k\audio\fold5\1...
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing,/tmp\sound_datasets\urbansound8k\audio\fold5\1...


### Load the Audio 

In [22]:
def load_audio(path):
    # Load the audio file
    # mono=TRUE ensures single-channel audio
    y, sample_rate = librosa.load(path, sr=SAMPLE_RATE, mono=True)

    # Duration of 4 seconds
    if len(y) > SAMPLES_PER_CLIP:
        y = y[:SAMPLES_PER_CLIP]     # cuts
    elif len(y) < SAMPLES_PER_CLIP:
        y = np.pad(y, (0, SAMPLES_PER_CLIP - len(y)))   # padding with zeros

    return y

### Feature Extraction

**Mel-Frequency Cepstral Coefficients (MFCCs) for the Recurrent Neural Network (RNN)**

In [31]:
def extract_mfcc_sequence(file_path):
    y = load_audio(file_path)

    # Calculating the MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)

    # Z-Score Normalization
    # For each MFCC coefficient, subtracts its average value across all time frames and then divides by its standard deviation.
    # We added 1e-8 to avoid division by zero
    mfcc = (mfcc - mfcc.mean(axis=1, keepdims=True)) / (mfcc.std(axis=1, keepdims=True) + 1e-8)

    # Transpose to (n_frames, n_mfcc)
    return mfcc.T

In [32]:
X_rnn_list = []   # MFCCs sequences
y_list = []       # Label
fold_list = []    # Fold 

for _, row in metadata.iterrows():
    mfcc_seq = extract_mfcc_sequence(row["file_path"])
    X_rnn_list.append(mfcc_seq)
    y_list.append(row["classID"])
    fold_list.append(row["fold"])

### Save Pre-Processed Features

In [33]:
# Create folder
PROCESSED_DIR = "processed_data"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Convert lists to numpy arrays
X_rnn_array = np.array(X_rnn_list, dtype=object)  # X_rnn_list has sequences with different sizes so, we use dtype=object
y_array = np.array(y_list, dtype=np.int64)
fold_array = np.array(fold_list, dtype=np.int64)

# Save to .npy files
np.save(os.path.join(PROCESSED_DIR, "X_rnn_mfcc.npy"), X_rnn_array)
np.save(os.path.join(PROCESSED_DIR, "y_labels.npy"), y_array)
np.save(os.path.join(PROCESSED_DIR, "folds.npy"), fold_array)